In [1]:
import json
with open('OpenAPIScripMaster.json','r') as f:
    data = json.load(f) 

In [2]:
for item in data:
    if item['name'] == 'INFY' and item['exch_seg'] == 'NSE':
        print(item['token'])

1594


In [3]:
targets = {"INFY", "HCLTECH", "SUNPHARMA"}
for item in data:
    if item['name'] in targets and item['exch_seg'] == 'NSE':
        print(f"{item['name']:<12} {item['token']:<8} {item['symbol']:<16} {item['exch_seg']}")

HCLTECH      7229     HCLTECH-EQ       NSE
INFY         1594     INFY-EQ          NSE
SUNPHARMA    3351     SUNPHARMA-EQ     NSE


In [ ]:
# Connect Storage Pipeline to Live Trading Node

Currently, the `trading` node receives market data from the Angel One WebSocket and pipes it directly into the NautilusTrader engine for live strategy execution, but the raw ticks are discarded from memory afterwards.

This plan wires your existing `storage` crate (which handles QuestDB and Parquet files) directly into the `trading` node.

## Proposed Changes

### 1. `trading/Cargo.toml`
Add the `storage` crate as a dependency so the trading node can access the storage consumer.

#### [MODIFY] trading/Cargo.toml
- Add `storage = { path = "../storage" }` to dependencies.

### 2. `adapter_angelone/src/config.rs`
Update the `AngelOneDataClientConfig` to accept an optional channel sender that will forward raw ticks out of the adapter.

#### [MODIFY] adapter_angelone/src/config.rs
- Add `pub tick_sender: Option<tokio::sync::mpsc::Sender<common::Tick>>` to `AngelOneDataClientConfig`.
- Update the `new()` constructor to accept this parameter.

### 3. `adapter_angelone/src/data.rs`
Modify the binary WebSocket frame handler. Every time a valid frame is parsed, convert it into a `common::Tick` and fire it down the channel.

#### [MODIFY] adapter_angelone/src/data.rs
- In `handle_binary_frame`, check if `tick_sender` is configured.
- If it is, call `packet.to_tick()` and send the raw tick to the channel. This happens independently of the Nautilus `DataEvent` routing.

### 4. `trading/src/main.rs`
Wire everything together at the orchestration level.

#### [MODIFY] trading/src/main.rs
- Create a `tokio::sync::mpsc::channel(8192)` for tick transport.
- Pass the `Sender` into `AngelOneDataClientConfig`.
- Spawn the storage consumer task: `tokio::spawn(storage::storage_consumer(rx));`

## User Review Required

> [!IMPORTANT]
> The `storage` crate requires a `./data/raw` folder to exist to write the Parquet files. I will ensure the trading node automatically creates this directory on startup if it doesn't exist.

## Verification Plan
1. Compile the workspace to ensure no typing errors between the adapter and the storage channels.
2. Ensure the bot still runs and successfully connects to Angel One.
3. Once data flows (when the market opens), QuestDB will populate on `localhost:9000` and Parquet files will begin generating in `./data/raw/`.
